# Laboratorio: PCA y aplicaciones

Centraremos datos, verificaremos la equivalencia distancia--varianza, visualizaremos la proyección de puntos de $\mathbb R^3$ sobre un plano principal y estudiaremos factores latentes en una matriz de calificaciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1. Datos tridimensionales y centrado

Las observaciones siguen aproximadamente un plano afín. PCA debe aplicarse después de restar la media.

In [ ]:
X = np.array([[ 3.2,  1.4,  3.1],
              [ 2.4,  0.9,  1.6],
              [-0.8, -1.3,  0.7],
              [ 0.5,  2.1,  1.3],
              [-1.2,  0.0,  0.9],
              [ 2.0, -1.0,  0.6],
              [-0.1,  1.5,  1.7],
              [ 1.1, -0.4,  0.4]])
media = X.mean(axis=0)
Z = X-media
assert np.allclose(Z.mean(axis=0), 0)
U, s, Vt = np.linalg.svd(Z, full_matrices=False)
V2 = Vt[:2].T
T2 = Z @ V2
Z2 = T2 @ V2.T
X2 = media + Z2
media, s, V2

## 2. Equivalencia numérica entre distancia y varianza

La identidad pitagórica afirma que la suma total de cuadrados es la suma de la variación representada en el plano y el error residual.

In [ ]:
suma_total = np.linalg.norm(Z, 'fro')**2
suma_proyectada = np.linalg.norm(T2, 'fro')**2
suma_distancias = np.linalg.norm(Z-Z2, 'fro')**2
varianzas = np.var(T2, axis=0, ddof=0)
assert np.allclose(suma_total, suma_proyectada+suma_distancias)
assert np.allclose(suma_proyectada/len(Z), varianzas.sum())
assert np.allclose(varianzas, s[:2]**2/len(Z))
suma_total, suma_proyectada, suma_distancias, varianzas

## 3. Gráfico de la proyección de $\mathbb R^3$ sobre el plano principal

El plano principal pasa por la media. Cada segmento punteado une un dato con su proyección y es ortogonal a ambas direcciones del plano.

In [ ]:
t = np.linspace(-3.2, 3.2, 16)
a, b = np.meshgrid(t, t)
plano = media + a[..., None]*V2[:, 0] + b[..., None]*V2[:, 1]
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(plano[..., 0], plano[..., 1], plano[..., 2],
                alpha=.22, color='tab:green')
ax.scatter(*X.T, s=48, color='tab:blue', label='datos')
ax.scatter(*X2.T, s=48, color='tab:orange', label='proyecciones')
for x, px in zip(X, X2):
    ax.plot(*np.vstack([x, px]).T, '--', color='tab:purple', lw=1.8)
    assert np.allclose(V2.T @ (x-px), 0)
ax.scatter(*media, s=80, marker='x', color='black', label='media')
ax.set(xlabel='$x_1$', ylabel='$x_2$', zlabel='$x_3$',
       title='PCA: plano principal y residuos ortogonales')
ax.legend()
plt.tight_layout();

## 4. Coordenadas principales y varianza explicada

El diagrama siguiente usa únicamente las dos coordenadas de cada punto.

In [ ]:
proporcion_varianza = s**2/np.sum(s**2)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(T2[:, 0], T2[:, 1], s=50)
for i, (a_i, b_i) in enumerate(T2):
    axes[0].annotate(str(i+1), (a_i, b_i), xytext=(4, 4),
                     textcoords='offset points')
axes[0].set(xlabel='primera componente', ylabel='segunda componente',
            title='Datos en dos coordenadas principales')
axes[0].axhline(0, color='0.8', lw=1)
axes[0].axvline(0, color='0.8', lw=1)
axes[1].bar(np.arange(1, len(s)+1), proporcion_varianza)
axes[1].set(xlabel='componente', ylabel='proporción de varianza',
            title='Varianza explicada por componente', xticks=range(1, len(s)+1))
plt.tight_layout()
proporcion_varianza, proporcion_varianza[:2].sum()

## 5. Matriz de calificaciones de películas

Aquí no centramos la matriz: buscamos una aproximación directa de calificaciones mediante dos factores latentes. Las filas son personas y las columnas son películas.

In [ ]:
personas = ['Abbie', 'Bailey', 'Catherine', 'Darlene',
            'Elena', 'Fatima', 'Gladys']
peliculas = ['Alien', 'Casablanca', 'Star Wars', 'Titanic', 'The Matrix']
A = np.array([[0, 2, 0, 2, 1],
              [1, 0, 1, 0, 1],
              [5, 0, 5, 0, 5],
              [0, 4, 0, 4, 2],
              [3, 0, 3, 0, 3],
              [0, 5, 0, 5, 0],
              [4, 0, 4, 0, 4]], dtype=float)
U_A, s_A, Vt_A = np.linalg.svd(A, full_matrices=False)
A2 = (U_A[:, :2]*s_A[:2]) @ Vt_A[:2]
assert np.allclose(np.linalg.norm(A-A2, 'fro'), np.sqrt(np.sum(s_A[2:]**2)))
s_A, np.linalg.norm(A-A2, 'fro'), A2

## 6. Mapa de personas y películas en dos factores

La predicción de una entrada puede escribirse como el producto interno entre el perfil de la persona $U_2\Sigma_2$ y el perfil de la película $V_2$. El signo simultáneo de ambos ejes es arbitrario.

In [ ]:
coords_personas = U_A[:, :2]*s_A[:2]
coords_peliculas = Vt_A[:2].T
assert np.allclose(A2, coords_personas @ coords_peliculas.T)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(*coords_personas.T, color='tab:blue')
for nombre, (x, y) in zip(personas, coords_personas):
    axes[0].annotate(nombre, (x, y), xytext=(4, 4), textcoords='offset points')
axes[0].set(title='Perfiles de personas', xlabel='factor 1', ylabel='factor 2')
axes[1].scatter(*coords_peliculas.T, color='tab:red')
for nombre, (x, y) in zip(peliculas, coords_peliculas):
    axes[1].annotate(nombre, (x, y), xytext=(4, 4), textcoords='offset points')
axes[1].set(title='Perfiles de películas', xlabel='factor 1', ylabel='factor 2')
for ax in axes:
    ax.axhline(0, color='0.8', lw=1); ax.axvline(0, color='0.8', lw=1)
plt.tight_layout();

## 7. Filtrado colaborativo: Hannah

Usamos las calificaciones conocidas de Hannah para estimar sus dos coordenadas y predecir su calificación de *The Matrix*. Se resuelve con los valores calculados, no con los redondeos mostrados en la hoja de teoría.

In [ ]:
indices_conocidos = [peliculas.index('Alien'), peliculas.index('Casablanca')]
calificaciones = np.array([4., 1.])
B = (s_A[:2, None]*Vt_A[:2, indices_conocidos]).T
hannah = np.linalg.solve(B, calificaciones)
indice_matrix = peliculas.index('The Matrix')
prediccion = hannah @ (s_A[:2]*Vt_A[:2, indice_matrix])
assert np.allclose(B @ hannah, calificaciones)
hannah, prediccion

## 8. Actividades

1. Compara el plano principal de $X$ con el subespacio óptimo calculado sin centrar.
2. Verifica que la suma de las varianzas sea la traza de la matriz de covarianza.
3. Reconstruye cada observación usando solo la primera componente y mide el error.
4. Interpreta, a partir de los signos y agrupamientos, los dos factores de las películas.
5. Cambia las dos calificaciones conocidas de Hannah y analiza la predicción.
6. Explica por qué tratar todos los ceros como datos faltantes cambiaría el problema.